In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
import warnings
import joblib
warnings.filterwarnings('ignore')

In [ ]:
start_day = 1223
end_day = 1307

def get_week_idx(day_num, start_day=1223):
    return (day_num - start_day) // 7

retail_files = []
retail_events_list = []

for day in range(start_day, end_day + 1):
    day_str = f"{day:05d}"
    file_pattern = f'dataset/small/retail/events/{day_str}.pq'
    files = glob.glob(file_pattern)
    
    for file in files:
        df = pd.read_parquet(file)
        df['idx_time'] = get_week_idx(day)  # Add week index
        retail_events_list.append(df)

retail_events = pd.concat(retail_events_list, ignore_index=True)

train_data = retail_events[
    retail_events['idx_time'].isin([0, 1, 2, 3, 4, 5, 6])
]

val_data = retail_events[
    retail_events['idx_time'].isin([8])
]

test_data = retail_events[
    retail_events['idx_time'].isin([10])
]

print("Retail weeks distribution:")
print(retail_events['idx_time'].value_counts().sort_index())

Retail weeks distribution:
idx_time
0     16641323
1     11069097
2     10040530
3     13165837
4     13352574
5     14470383
6     11065028
7     13415471
8     11185032
9     12879972
10    12258870
11    13806665
12     1782374
Name: count, dtype: int64


In [4]:
retail_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,idx_time
0,1223 days 00:00:00.002308,25251102,fmcg_250797,catalog,view,android,0
1,1223 days 00:00:00.290562,23029040,fmcg_90081,main,view,android,0
2,1223 days 00:00:00.378700,70355590,fmcg_1041263,catalog,view,android,0
3,1223 days 00:00:00.461884,70376079,fmcg_911625,catalog,view,android,0
4,1223 days 00:00:00.634998,7521752,fmcg_843098,catalog,view,android,0


In [5]:
retail_events['subdomain'].value_counts()

subdomain
catalog    133945346
search      13445023
other        3685239
main         2058940
i2i          1998608
Name: count, dtype: int64

In [6]:
retail_events['action_type'].value_counts()

action_type
view             148196142
order              3685239
added_to_cart      2049036
click              1202739
Name: count, dtype: int64

In [7]:
train_data['idx_item'] = train_data['item_id'].str.split('_').str[1]
train_data['item_id'] = train_data['item_id'].str.split('_').str[0]
train_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
train_data['treated'] = train_data['treated'].isin(['i2i']).astype(int)
train_data['outcome'] = train_data['outcome'].isin(['click', 'added_to_cart', 'order']).astype(int)
train_data.drop(['os'], axis = 1, inplace = True)
train_data['personal_popular'] = train_data.groupby(['idx_user', 'idx_item'])['outcome'].transform('sum')
train_data['personal_popular'].fillna(0, inplace=True)
train_data = train_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
train_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,25251102,250797,fmcg,0,0,0,1223 days 00:00:00.002308,0
1,23029040,90081,fmcg,0,0,0,1223 days 00:00:00.290562,0
2,70355590,1041263,fmcg,0,0,0,1223 days 00:00:00.378700,0
3,70376079,911625,fmcg,0,0,0,1223 days 00:00:00.461884,0
4,7521752,843098,fmcg,0,0,1,1223 days 00:00:00.634998,0


In [8]:
df_merge = train_data[['idx_user', 'idx_item', 'personal_popular']]

In [9]:
test_data['idx_item'] = test_data['item_id'].str.split('_').str[1]
test_data['item_id'] = test_data['item_id'].str.split('_').str[0]
test_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
test_data['treated'] = test_data['treated'].isin(['i2i']).astype(int)
test_data['outcome'] = test_data['outcome'].isin(['click', 'added_to_cart', 'order']).astype(int)
test_data.drop(['os'], axis = 1, inplace = True)
test_data = pd.merge(test_data, df_merge, on=['idx_user', 'idx_item'], how='left')
test_data['personal_popular'].fillna(0, inplace=True)
test_data = test_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
test_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,70995916,205205,fmcg,0,0,0.0,1293 days 00:00:00.011863,10
1,49107917,900072,fmcg,0,0,0.0,1293 days 00:00:00.061047,10
2,61815931,881658,fmcg,0,0,0.0,1293 days 00:00:00.126640,10
3,19386390,276028,fmcg,0,0,0.0,1293 days 00:00:00.280741,10
4,38541895,402995,fmcg,0,0,0.0,1293 days 00:00:00.352384,10


In [10]:
val_data['idx_item'] = val_data['item_id'].str.split('_').str[1]
val_data['item_id'] = val_data['item_id'].str.split('_').str[0]
val_data.rename(columns={'user_id': 'idx_user', 'subdomain': 'treated', 'action_type': 'outcome'}, inplace=True)
val_data['treated'] = val_data['treated'].isin(['i2i']).astype(int)
val_data['outcome'] = val_data['outcome'].isin(['click', 'added_to_cart', 'order']).astype(int)
val_data.drop(['os'], axis = 1, inplace = True)
val_data = pd.merge(val_data, df_merge, on=['idx_user', 'idx_item'], how='left')
val_data['personal_popular'].fillna(0, inplace=True)
val_data = val_data[['idx_user', 'idx_item', 'item_id', 'treated', 'outcome', 'personal_popular', 'timestamp', 'idx_time']]
val_data.head()

,idx_user,idx_item,item_id,treated,outcome,personal_popular,timestamp,idx_time
0,85423355,582158,fmcg,0,0,0.0,1279 days 00:00:00.212758,8
1,59048330,976287,fmcg,0,0,0.0,1279 days 00:00:00.465426,8
2,425340,969112,fmcg,0,0,0.0,1279 days 00:00:00.618009,8
3,66938590,609785,fmcg,0,0,0.0,1279 days 00:00:00.672016,8
4,54469733,665118,fmcg,0,1,0.0,1279 days 00:00:01.022620,8


In [11]:
train_data = train_data[train_data['idx_time'] == 0]
test_data['idx_time'] = 0
val_data['idx_time'] = 0
use_trained_models = True

In [12]:
len(train_data)

16641323

In [13]:
len(test_data)

31182621

In [14]:
len(val_data)

29180820

In [ ]:
print("Training DR-Learner...")

df_with_features_train = train_data.copy()

df_with_features_train['hour'] = df_with_features_train['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_train.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_train.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_train = df_with_features_train.merge(item_popularity, on='idx_item', how='left')
df_with_features_train = df_with_features_train.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_train.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_train = df_with_features_train.merge(user_activity, on='idx_user', how='left')

feature_cols = [
    'hour', 
    'item_popularity', 
    'item_cvr',
    'user_unique_items',
    'user_total_interactions',
    'user_conversion_rate'
]

print("Step 1: Preparing training data...")

X_train = df_with_features_train[feature_cols].fillna(0).values
y_train = df_with_features_train['outcome'].values
t_train = df_with_features_train['treated'].values

print(f"Training samples: {len(X_train)}")
print(f"Treated rate: {t_train.mean():.4f}")
print(f"Outcome rate: {y_train.mean():.4f}")

if not use_trained_models:
    print("Step 2: Cross-fitting to compute DR pseudo-outcomes...")

    n_folds = 3
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    mu0_cf = np.zeros_like(y_train, dtype=float)
    mu1_cf = np.zeros_like(y_train, dtype=float)
    propensity_cf = np.zeros_like(y_train, dtype=float)

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"  Fold {fold + 1}/{n_folds}")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        t_tr, t_val = t_train[train_idx], t_train[val_idx]
        
        prop_model = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')
        prop_model.fit(X_tr, t_tr)
        propensity_cf[val_idx] = prop_model.predict_proba(X_val)[:, 1]
        
        control_mask = (t_tr == 0)
        if control_mask.sum() > 0:
            model_control = GradientBoostingRegressor(
                n_estimators=50, max_depth=5, min_samples_leaf=50, 
                learning_rate=0.05, random_state=42
            )
            model_control.fit(X_tr[control_mask], y_tr[control_mask])
            mu0_cf[val_idx] = model_control.predict(X_val)
        else:
            mu0_cf[val_idx] = 0
        
        treated_mask = (t_tr == 1)
        if treated_mask.sum() > 0:
            model_treated = GradientBoostingRegressor(
                n_estimators=50, max_depth=5, min_samples_leaf=50,
                learning_rate=0.05, random_state=42
            )
            model_treated.fit(X_tr[treated_mask], y_tr[treated_mask])
            mu1_cf[val_idx] = model_treated.predict(X_val)
        else:
            mu1_cf[val_idx] = 0

    propensity_cf = np.clip(propensity_cf, 0.0001, 0.9999)

    dr_pseudo = np.where(
        t_train == 1,
        mu1_cf + (y_train - mu1_cf) / propensity_cf,
        mu0_cf + (y_train - mu0_cf) / (1 - propensity_cf)
    )

    dr_pseudo = np.clip(dr_pseudo, -1, 1)

    print(f"  DR pseudo-outcome range: [{dr_pseudo.min():.4f}, {dr_pseudo.max():.4f}]")

    print("Step 3: Training final DR model...")

    dr_final_model = GradientBoostingRegressor(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=50,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    )

    dr_final_model.fit(X_train, dr_pseudo)

else:
    dr_final_model = joblib.load("model_dr_learner.joblib")

train_causal_effect = dr_final_model.predict(X_train)
train_data['causal_effect'] = train_causal_effect

print(f"  Train causal effect range: [{train_causal_effect.min():.4f}, {train_causal_effect.max():.4f}]")
print(f"  Train causal effect mean: {train_causal_effect.mean():.4f}")

if not use_trained_models:
    print("Step 4: Training final models for prediction...")

    final_propensity_model = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')
    final_propensity_model.fit(X_train, t_train)

else:
    final_propensity_model = joblib.load("model_propensity.joblib")

train_data['propensity'] = final_propensity_model.predict_proba(X_train)[:, 1]
train_data['propensity'] = np.clip(train_data['propensity'], 0.0001, 0.9999)

print("  Final models trained successfully")

Training DR-Learner...
Step 1: Preparing training data...
Training samples: 16641323
Treated rate: 0.0093
Outcome rate: 0.0366
  Train causal effect range: [-0.8414, 1.1466]
  Train causal effect mean: -0.0014
  Final models trained successfully


In [ ]:
df_with_features_val = val_data.copy()

df_with_features_val['hour'] = df_with_features_val['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_val.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_val.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_val = df_with_features_val.merge(item_popularity, on='idx_item', how='left')
df_with_features_val = df_with_features_val.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_val.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_val = df_with_features_val.merge(user_activity, on='idx_user', how='left')

df_with_features_val[feature_cols] = df_with_features_val[feature_cols].fillna(0)

X_val = df_with_features_val[feature_cols].fillna(0).values

print("Step 5: Applying to validation data...")

val_propensity = final_propensity_model.predict_proba(X_val)[:, 1]
val_propensity = np.clip(val_propensity, 0.0001, 0.9999)
val_data['propensity'] = val_propensity

val_causal_effect = dr_final_model.predict(X_val)
val_data['causal_effect'] = val_causal_effect

print(f"  Val causal effect range: [{val_causal_effect.min():.4f}, {val_causal_effect.max():.4f}]")
print(f"  Val causal effect mean: {val_causal_effect.mean():.4f}")

Step 5: Applying to validation data...
  Val causal effect range: [-0.9374, 1.1942]
  Val causal effect mean: 0.0188


In [ ]:
print("Step 6: Applying to test data...")

df_with_features_test = test_data.copy()

df_with_features_test['hour'] = df_with_features_test['timestamp'].dt.seconds // 3600

item_popularity = df_with_features_test.groupby('idx_item')['idx_user'].nunique().reset_index()
item_popularity.columns = ['idx_item', 'item_popularity']

item_cvr = df_with_features_test.groupby('idx_item')['outcome'].mean().reset_index()
item_cvr.columns = ['idx_item', 'item_cvr']

df_with_features_test = df_with_features_test.merge(item_popularity, on='idx_item', how='left')
df_with_features_test = df_with_features_test.merge(item_cvr, on='idx_item', how='left')

user_activity = df_with_features_test.groupby('idx_user').agg({
    'outcome': 'sum',
    'idx_item': 'nunique',
    'timestamp': 'count'
}).reset_index()
user_activity.columns = ['idx_user', 'user_total_outcomes', 'user_unique_items', 'user_total_interactions']
user_activity['user_conversion_rate'] = user_activity['user_total_outcomes'] / user_activity['user_total_interactions']

df_with_features_test = df_with_features_test.merge(user_activity, on='idx_user', how='left')

df_with_features_test[feature_cols] = df_with_features_test[feature_cols].fillna(0)

X_test = df_with_features_test[feature_cols].fillna(0).values

test_propensity = final_propensity_model.predict_proba(X_test)[:, 1]
test_propensity = np.clip(test_propensity, 0.0001, 0.9999)
test_data['propensity'] = test_propensity

test_causal_effect = dr_final_model.predict(X_test)
test_data['causal_effect'] = test_causal_effect

print(f"  Test causal effect range: [{test_causal_effect.min():.4f}, {test_causal_effect.max():.4f}]")
print(f"  Test causal effect mean: {test_causal_effect.mean():.4f}")

Step 6: Applying to test data...
  Test causal effect range: [-0.9443, 1.1827]
  Test causal effect mean: 0.0124


In [ ]:
print("Step 7: Saving models and data...")

joblib.dump(dr_final_model, 'model_dr_learner.joblib')
joblib.dump(final_propensity_model, 'model_propensity.joblib')

print("  All files saved successfully!")

Step 7: Saving models and data...
  All files saved successfully!


In [18]:
train_data['causal_effect'].describe()

count    1.664132e+07
mean    -1.414466e-03
std      1.214201e-01
min     -8.413911e-01
25%     -2.764418e-03
50%      6.529822e-04
75%      1.654275e-03
max      1.146582e+00
Name: causal_effect, dtype: float64

In [19]:
val_data['causal_effect'].describe()

count    2.918082e+07
mean     1.881030e-02
std      1.924429e-01
min     -9.373903e-01
25%     -2.210325e-03
50%      6.169657e-04
75%      1.795729e-03
max      1.194191e+00
Name: causal_effect, dtype: float64

In [20]:
test_data['causal_effect'].describe()

count    3.118262e+07
mean     1.238254e-02
std      1.705406e-01
min     -9.443492e-01
25%     -2.272702e-03
50%      6.021483e-04
75%      1.726052e-03
max      1.182733e+00
Name: causal_effect, dtype: float64

In [ ]:
print("Step 8: Applying discretization (optional)...")

train_data['causal_effect'] = train_data['causal_effect'].apply(
    lambda x: 1 if x > 0.5 else (-1 if x < -0.5 else 0)
)

test_data['causal_effect'] = test_data['causal_effect'].apply(
    lambda x: 1 if x > 0.5 else (-1 if x < -0.5 else 0)
)

val_data['causal_effect'] = val_data['causal_effect'].apply(
    lambda x: 1 if x > 0.5 else (-1 if x < -0.5 else 0)
)

print(train_data['causal_effect'].value_counts())

Step 8: Applying discretization (optional)...
causal_effect
 0    16428844
 1      208480
-1        3999
Name: count, dtype: int64


In [25]:
train_data.to_csv('data_train.csv', index=False)
val_data.to_csv('data_vali.csv', index=False)
test_data.to_csv('data_test.csv', index=False)

## Looking at max and min causal_effect that can be obtained

In [2]:
test_data = pd.read_csv('data_test.csv')
test_data_descending = test_data.sort_values(by=['idx_user', 'causal_effect'], ascending=False)
test_data_ascending = test_data.sort_values(by=['idx_user', 'causal_effect'], ascending=True)

In [3]:
df_ranking_head_10 = test_data_descending.groupby('idx_user').head(10)
df_ranking_tail_10 = test_data_ascending.groupby('idx_user').head(10)

print(f"Best CP@10: {np.nanmean(df_ranking_head_10.loc[:, 'causal_effect'].values):.6f}")
print(f"Worst CP@10: {np.nanmean(df_ranking_tail_10.loc[:, 'causal_effect'].values):.6f}")

Best CP@10: 0.639515
Worst CP@10: 0.389102


In [4]:
df_ranking_head_100 = test_data_descending.groupby('idx_user').head(100)
df_ranking_tail_100 = test_data_ascending.groupby('idx_user').head(100)

print(f"Best CP@100: {np.nanmean(df_ranking_head_100.loc[:, 'causal_effect'].values):.6f}")
print(f"Worst CP@100: {np.nanmean(df_ranking_tail_100.loc[:, 'causal_effect'].values):.6f}")

Best CP@100: 0.427432
Worst CP@100: 0.317024


In [5]:
def dcg_at_k(rank_k, x):
    k = min(rank_k, len(x)) 
    return np.sum(x[:k] / np.log2(np.arange(k) + 2))

In [6]:
print(f"Best CDCG: {float(np.nanmean(test_data_descending.groupby('idx_user')['causal_effect'].apply(lambda x: dcg_at_k(100000, x)))):.6f}")
print(f"Worst CDCG: {float(np.nanmean(test_data_ascending.groupby('idx_user')['causal_effect'].apply(lambda x: dcg_at_k(100000, x)))):.6f}")

Best CDCG: 10.680209
Worst CDCG: 9.104545
